# Automated AI Data Scientist Report

**Dataset:** `iris`  
**Rows:** 150  
**Columns:** 5  
**Inferred target:** `species`  
**Problem type:** `classification`

This notebook was auto-generated to mirror a professional EDA + feature engineering workflow with business-facing commentary.

## Problem Understanding

This notebook follows an end-to-end workflow:
1. Load the dataset
2. Audit quality issues
3. Clean and preprocess the data
4. Engineer useful features
5. Run deep EDA
6. Summarize business insights and modeling recommendations

The system inferred **`species`** as the likely target with confidence **1.00**.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from ai_data_scientist.data_loader import DatasetLoader
from ai_data_scientist.preprocessing import DataCleaner
from ai_data_scientist.eda import EDAEngine
from ai_data_scientist.visualization import Visualizer
from ai_data_scientist.imbalance_handler import ImbalanceHandler

In [ ]:
DATASET_PATH = r"/mnt/user-data/outputs/ai_data_scientist_project/sample_data/iris.csv"
TARGET_OVERRIDE = 'species'
loader = DatasetLoader()
cleaner = DataCleaner()
eda_engine = EDAEngine()
visualizer = Visualizer()
imbalance_handler = ImbalanceHandler()

## 1. Data Loading

In [ ]:
df = loader.load(DATASET_PATH)
profile = loader.profile(df, DATASET_PATH, target_column=TARGET_OVERRIDE)

print("Dataset shape:", df.shape)
print("Target column:", profile.target_column)
print("Problem type:", profile.problem_type)
print("Numeric columns:", profile.numeric_columns[:10])
print("Categorical columns:", profile.categorical_columns[:10])
print("Datetime columns:", profile.datetime_columns[:10])
print("Text columns:", profile.text_columns[:10])

df.head()

### Initial Dataset Snapshot

- Memory footprint: **0.01 MB**
- Duplicate rows: **0**
- Missing cells: **0**
- Numeric features: **4**
- Categorical features: **1**
- Datetime features: **0**
- Text features: **0**

**Recommended starter models:** LogisticRegression, RandomForestClassifier, HistGradientBoostingClassifier.

## 2. Data Cleaning & Preprocessing

In [ ]:
cleaning_result = cleaner.clean(df, profile)
clean_df = cleaning_result.cleaned_df
cleaning_report = cleaning_result.report

print(cleaning_report)
clean_df.head()

### Cleaning Summary

- Original shape: **(150, 5)**
- Final shape: **(149, 5)**
- Rows removed: **1**
- Duplicate rows removed: **1**

**Missing-value handling**
- No imputation was necessary.

**Outlier handling**
- `sepal_width_(cm)` had 4 IQR outliers and was capped to [2.2, 4.152000000000001]

## 3. Feature Engineering

In [ ]:
recommendations = cleaner.build_preprocessing_recommendations(clean_df, profile)
print("Preprocessing recommendations:")
print(recommendations)

clean_df.head()

### Feature Engineering Notes

- No additional datetime or text features were required for this dataset.

## 4. Exploratory Data Analysis (EDA)

In [ ]:
eda_summary = eda_engine.summarize(clean_df, profile)
print("Dataset overview:")
print(eda_summary["dataset_overview"])

visualizer.plot_missing_values(clean_df)
visualizer.plot_numeric_distributions(clean_df, clean_df.select_dtypes(include="number").columns[:6])
visualizer.plot_boxplots(clean_df, clean_df.select_dtypes(include="number").columns[:6])
visualizer.plot_categorical_counts(clean_df, [c for c in clean_df.columns if clean_df[c].dtype == "object"][:6])
visualizer.plot_correlation_heatmap(clean_df)
visualizer.plot_pairplot(clean_df, clean_df.select_dtypes(include="number").columns[:5], target_col=profile.target_column)
visualizer.plot_target_relationships(clean_df, profile.target_column)
visualizer.plot_interactive_summary(clean_df, profile.target_column)

### Key Insights

- After cleaning, the dataset has no materially problematic missing-value concentrations.
- The strongest numeric relationship is petal_length_(cm) vs petal_width_(cm) with correlation 0.963. This is a strong candidate for multicollinearity checks or interaction features.
- Target classes are reasonably balanced, so standard stratified training should work well.

**Top correlation pairs**
- petal_length_(cm) vs petal_width_(cm): 0.963
- sepal_length_(cm) vs petal_length_(cm): 0.874
- sepal_length_(cm) vs petal_width_(cm): 0.821
- sepal_width_(cm) vs petal_length_(cm): -0.429
- sepal_width_(cm) vs petal_width_(cm): -0.366

## 5. Target Analysis & Imbalance Handling

In [ ]:
if profile.target_column and profile.problem_type == "classification":
    detection = imbalance_handler.detect(clean_df, profile.target_column)
    print("Imbalance detection:", detection)
    rebalance_report = imbalance_handler.rebalance_train_split(clean_df, profile.target_column, profile)
    print("Rebalancing report:", rebalance_report)
else:
    print("Imbalance handling skipped because no classification target was detected.")

### Imbalance Review

- Sampler used: **SMOTE**
- Before balancing: **{"setosa": 40, "virginica": 39, "versicolor": 40}**
- After balancing: **{"setosa": 40, "virginica": 40, "versicolor": 40}**
- Train shape before: **(119, 4)**
- Train shape after: **(120, 4)**

## 6. Model Suggestions & Next Steps

In [ ]:
print("Recommended starter models:")
print(eda_summary["recommended_models"])

if profile.target_column:
    print("Top target summary:")
    print(eda_summary["target_summary"])
else:
    print("No target summary available.")

## 7. Conclusion

This auto-generated report identified the main structure of the dataset, cleaned the core quality issues, surfaced useful features, and summarized the most actionable EDA findings.

For modeling, start with: **LogisticRegression, RandomForestClassifier, HistGradientBoostingClassifier**. Track data leakage, validate transformations in cross-validation, and document business assumptions before production deployment.

If the inferred target **`species`** is incorrect, rerun the pipeline with an explicit `--target` argument.